# MICS Offline Analysis

Assemble MICS behavioral data offline. The **Elasticsearch events layer always
works on its own**; the Open Ephys and spike layers attach automatically only
when a recording exists for that session.

This one notebook replaces the per-session legacy notebooks: add a session to
the `SESSIONS` block below and run the sections you need.

## How to add a session

Edit the `SESSIONS` list. Minimum fields per entry:

- `key` — short id like `"m74s4"` (subject `m74`, session `4`)
- `es_subject` — the **descriptive** Elasticsearch subject string (e.g.
  `"m74_cue_reward"`), *not* the key
- `es_session` — the ES session number

Everything else is derived by convention from the sessions share:

- ephys folder `<subject>s<n>_<date>_<time>` → auto-found, with its `Record Node`
- spikes → loose `spikes_<subject>s<n>.pkl`, else inline `processed.xls`

Optional overrides: `es_host`, `es_index`, `run_id`, `ephys_folder`,
`record_node`, `spike_path`, `trigger_channel`, `ttl_channel`, `sampling_rate`.

## How to run

Sections are independent and build up the picture:

1. **Events** (Elasticsearch) — always available, no ephys needed
2. **Open Ephys** — TTL/trigger edges (skips itself when no recording)
3. **Alignment & Spikes** — unify onto the ephys clock (graceful when absent)
4. **Figure** — raster + PSTH around a task event

A session with no ephys recording still produces a valid events-only result.

In [ ]:
import sys
sys.path.insert(0, "../src")  # harmless if `mics` is pip-installed (-e .)

from mics.config import SessionConfig
from mics.resolve import load_sessions
from mics.elastic import ElasticClient
from mics.cache import cached_pickle

## Find your subject + session (optional)

ES subjects are descriptive strings (e.g. `m74_cue_reward`), not the short key.
Look one up instead of memorizing it, then copy it into `SESSIONS` below.

In [ ]:
es = ElasticClient()                  # primary host
es.subjects("m74")                    # -> ['m74_cue_reward', 'm74_appetitive', ...]

In [ ]:
es.sessions("m74_cue_reward")         # -> {1: 960, 2: 781, 4: 2137, ...} (session -> #events)

In [ ]:
SESSIONS = [
    SessionConfig(key="m74s4", es_subject="m74_cue_reward", es_session=4),
    # SessionConfig(key="m74s1", es_subject="m74_cue_reward", es_session=1),
]

In [ ]:
sessions = load_sessions(SESSIONS)
for s in sessions:
    print(s.summary())

## Section 1 · Behavioral Events (Elasticsearch)

The always-available base layer — no ephys required. Key columns:

- `time` — **canonical event clock**: the precise on-Pi GPIO time where the event
  reports it, else the ES ingest time as fallback. Use this, not the ingest time.
- `pi_time` — precise GPIO time (null when the event doesn't report one)
- `raw_time` — ES ingest time, ~tens of ms after `pi_time` (provenance only)
- `relative_time` — seconds from session start, on the canonical `time` clock
- `run_id`/`subjects` — null for legacy data · `event_data` — raw dict

Results are cached per session key under `cache/<key>/events.pkl`; pass
`force=True` to refetch.

In [ ]:
events = {}
for s in sessions:
    events[s.key] = cached_pickle(
        s.key, "events",
        lambda s=s: ElasticClient(s.es_host).fetch_events(s),
    )
    print(f"{s.key}: {len(events[s.key])} events")

In [ ]:
key = sessions[0].key
df = events[key]
display(df.head(20))
df.event_type.value_counts()